# Customer Analysis 

## Objective 

The objective of this notebook is to analyze customer purchasing behavior and identify valuable customer segments using the RFM model.

The RFM model evaluate customers based on: 

- Recency
- Frequency
- Monetary Value 

The insight will support customer retention and marketing strategies.

In [1]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

Matplotlib is building the font cache; this may take a moment.


In [5]:
# Load Cleaned Dataset
df=pd.read_csv("../data/processed/online_retail_clean.csv")

In [6]:
# Converting InvoiceDate to datetime format
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [7]:
sales_df = df[
    (df["Quantity"]>0) &
    (df["Price"]>0)
].copy()

sales_df["Sales"] =(
sales_df["Quantity"] * 
sales_df["Price"]
)

In [8]:
sales_df.info()
sales_df.head()

<class 'pandas.DataFrame'>
Index: 1007914 entries, 0 to 1028000
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1007914 non-null  str           
 1   StockCode    1007914 non-null  str           
 2   Description  1007914 non-null  str           
 3   Quantity     1007914 non-null  int64         
 4   InvoiceDate  1007914 non-null  datetime64[us]
 5   Price        1007914 non-null  float64       
 6   Customer ID  779425 non-null   float64       
 7   Country      1007914 non-null  str           
 8   Sales        1007914 non-null  float64       
dtypes: datetime64[us](1), float64(3), int64(1), str(4)
memory usage: 76.9 MB


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Sales
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0


In [12]:
# Set Reference Date

reference_date = sales_df["InvoiceDate"].max() + pd.Timedelta(days=1)

print(reference_date)

2011-12-10 12:50:00


In [ ]:
# Aggregating sales by customer ID

rfm = sales_df.groupby("Customer ID").agg({
    "InvoiceDate": lambda x: (reference_date - x.max()).days,
    "Invoice": "nunique",
    "Sales": "sum"
})

In [14]:
rfm.head()

,InvoiceDate,Invoice,Sales
Customer ID,,,
12346.0,326,12,77556.46
12347.0,2,8,4921.53
12348.0,75,5,2019.40
12349.0,19,4,4428.69
12350.0,310,1,334.40


In [15]:
rfm.columns = [
    "Customer ID",
    "Recency",
    "Frequency",
    "Monetary"
]

ValueError: Length mismatch: Expected axis has 3 elements, new values have 4 elements

In [16]:
rfm = rfm.reset_index()

In [17]:
rfm.columns = [
    "Customer ID",
    "Recency",
    "Frequency",
    "Monetary"
]

In [18]:
rfm.head()

,Customer ID,Recency,Frequency,Monetary
0,12346.0,326,12,77556.46
1,12347.0,2,8,4921.53
2,12348.0,75,5,2019.40
3,12349.0,19,4,4428.69
4,12350.0,310,1,334.40


In [19]:
rfm.info()

<class 'pandas.DataFrame'>
RangeIndex: 5878 entries, 0 to 5877
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Customer ID  5878 non-null   float64
 1   Recency      5878 non-null   int64  
 2   Frequency    5878 non-null   int64  
 3   Monetary     5878 non-null   float64
dtypes: float64(2), int64(2)
memory usage: 183.8 KB


In [20]:
rfm.describe()

,Customer ID,Recency,Frequency,Monetary
count,5878.000000,5878.000000,5878.000000,5878.000000
mean,15315.313542,201.331916,6.289384,2955.904095
std,1715.572666,209.338707,13.009406,14440.852688
min,12346.000000,1.000000,1.000000,2.950000
25%,13833.250000,26.000000,1.000000,342.280000
50%,15314.500000,96.000000,3.000000,867.740000
75%,16797.750000,380.000000,7.000000,2248.305000
max,18287.000000,739.000000,398.000000,580987.040000


### Interpretation

- The dataset contains 5,878 unique customers with complete RFM information.
- The median Recency is 96 days, while 25% of customers have not purchased for more than 380 days, indicating a substantial inactive customer segment.
- Frequency is highly skewed, with 25% of customers making only one purchase, while a few customers purchased hundreds of times.
- These finding support the use of RFM scoring to identify valuable customers and prioritize retention or reactivation strategies.

In [21]:
# Creating R (Recency) Score

rfm["R_Score"] = pd.qcut(
    rfm["Recency"],
    q=5, 
    labels = [5, 4, 3, 2, 1]
)

In [22]:
# Checking R Score on the RFM table

rfm.head()

,Customer ID,Recency,Frequency,Monetary,R_Score
0,12346.0,326,12,77556.46,2
1,12347.0,2,8,4921.53,5
2,12348.0,75,5,2019.40,3
3,12349.0,19,4,4428.69,5
4,12350.0,310,1,334.40,2


In [ ]:
# Creating F Score on the RFM table
rfm["F_Score"] = pd.qcut(
    rfm["Frequency"],
    q=5,
    labels=[1, 2, 3, 4, 5]
)

ValueError: Bin edges must be unique: Index([1.0, 1.0, 2.0, 4.0, 8.0, 398.0], dtype='float64', name='Frequency').
You can drop duplicate edges by setting the 'duplicates' kwarg

In [26]:
# Recreating F score to avoide too many duplicates

rfm["F_Score"]=pd.qcut(
    rfm["Frequency"].rank(method="first"),
    q=5,
    labels= [1, 2, 3, 4, 5]
)


In [27]:
# Checking F Score
rfm.head()

,Customer ID,Recency,Frequency,Monetary,R_Score,F_Score
0,12346.0,326,12,77556.46,2,5
1,12347.0,2,8,4921.53,5,4
2,12348.0,75,5,2019.40,3,4
3,12349.0,19,4,4428.69,5,3
4,12350.0,310,1,334.40,2,1


## Frequecy Scoring Method

Frequency contains many duplicate values (especially one time purchasers). To ensure stable quantitle-based scoring, ranking was applied before using 'pd.qcut()'.

In [28]:
# Creating Monetary Score
rfm["M_Score"] = pd.qcut(
    rfm["Monetary"].rank(method="first"),
    q=5,
    labels=[1, 2, 3, 4, 5]
)

In [29]:
rfm.head()

,Customer ID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score
0,12346.0,326,12,77556.46,2,5,5
1,12347.0,2,8,4921.53,5,4,5
2,12348.0,75,5,2019.40,3,4,4
3,12349.0,19,4,4428.69,5,3,5
4,12350.0,310,1,334.40,2,1,2
